# SVCM — ITI-99 — Validation de code

**Profil** : IHE ITI SVCM
**Acteur testé** : SVCM-Terminology Consumer
**Transaction** : SVCM-ITI-99
**Référence Gazelle** : https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12210&cid=48839

Trois scénarios :
1. GET `$validate-code` — code `SA01` dans `JDV-J02-XdsHealthcareFacilityTypeCode-CISIS` (attendu : valide)
2. POST `$validate-code` — code `SA03` dans le même ValueSet (attendu : valide)
3. GET `$validate-code` — code `SA100` dans `TRE-R02-SecteurActivite` (attendu : **invalide**)

## Configuration

In [21]:
import requests
import json
import time
import os, getpass
from datetime import datetime
from urllib.parse import quote

FHIR_BASE = "https://gazelle-proxypatans.kereval.cloud:10102/fhir"
#FHIR_BASE = os.environ.get("FHIR_BASE", "https://smt.esante.gouv.fr/fhir")

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_POST = {
    "Accept": "application/fhir+json",
    "Content-Type": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_API = {
    "accept": "*/*",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}

# ── run output directory ──────────────────────────────────────────────────────
NOTEBOOK_ID = "06_iti99_validate"
RUN_TS  = datetime.now().strftime("%Y%m%dT%H%M%S")
RUN_DIR = os.path.join("runs", f"{NOTEBOOK_ID}_{RUN_TS}")
os.makedirs(RUN_DIR, exist_ok=True)

# ── helpers ───────────────────────────────────────────────────────────────────

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def http_post(url, body=None, headers=None):
    if headers is None:
        headers = HEADERS_POST
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → POST {url}")
            r = requests.post(url, json=body, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def save_artifact(step, filename, data, binary=False):
    """Save a test artifact to the run directory, prefixed with the step number."""
    path = os.path.join(RUN_DIR, f"step{step:03d}_{filename}")
    if binary:
        with open(path, "wb") as f:
            f.write(data)
    else:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ✓ {path}")
    return path

def print_keys(data, *keys):
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))

print(f"Configuration OK — sortie dans : {RUN_DIR}")


Configuration OK — sortie dans : runs/06_iti99_validate_20260311T212121


---
## Étapes 0–30 — Validation GET : SA01 dans JDV-J02 (code valide)

**Requête** : `GET /fhir/ValueSet/$validate-code?url=...JDV-J02...&code=SA01&_format=json&inferSystem=true`

In [22]:
# Étape 0  — Construction de la requête
JDV_J02_URL = "https://mos.esante.gouv.fr/NOS/JDV_J02-XdsHealthcareFacilityTypeCode-CISIS/FHIR/JDV-J02-XdsHealthcareFacilityTypeCode-CISIS"
TRE_R02_URL = "https://mos.esante.gouv.fr/NOS/TRE_R02-SecteurActivite/FHIR/TRE-R02-SecteurActivite"

# Étape 10 — TRANSACTION : GET $validate-code SA01
url_vc = f"{FHIR_BASE}/ValueSet/$validate-code?url={quote(JDV_J02_URL)}&code=SA01&_format=json&inferSystem=true"
r_vc = http_get(url_vc)
vc = r_vc.json()
save_artifact(30, "validate-sa01-result.json", vc)

# Étape 20 — Réception et intégration
params  = {p["name"]: p for p in vc.get("parameter", [])}
result  = params.get("result",  {}).get("valueBoolean")
display = params.get("display", {}).get("valueString", "")

# Étape 30 — PREUVE
print("[PREUVE étape 30] $validate-code SA01 (GET) :")
print(f"  result  : {result}  (attendu: True)")
print(f"  display : {display}  (attendu: Etablissement public de santé)")
assert result is True, f"Validation échouée pour SA01"
print("  ✓ Code valide")


  → GET https://gazelle-proxypatans.kereval.cloud:10102/fhir/ValueSet/$validate-code?url=https%3A//mos.esante.gouv.fr/NOS/JDV_J02-XdsHealthcareFacilityTypeCode-CISIS/FHIR/JDV-J02-XdsHealthcareFacilityTypeCode-CISIS&code=SA01&_format=json&inferSystem=true
  ✓ runs/06_iti99_validate_20260311T212121/step030_validate-sa01-result.json
[PREUVE étape 30] $validate-code SA01 (GET) :
  result  : True  (attendu: True)
  display : Etablissement public de santé  (attendu: Etablissement public de santé)
  ✓ Code valide


---
## Étapes 40–70 — Validation POST : SA03 dans JDV-J02 (code valide)

**Requête** : `POST /fhir/ValueSet/$validate-code`

In [23]:
# Étape 40 — Construction de la requête
# Étape 50 — TRANSACTION : POST $validate-code SA03 (Content-Type: application/json)
url_vc_post = f"{FHIR_BASE}/ValueSet/$validate-code"
vc_body = {
    "resourceType": "Parameters",
    "parameter": [
        {"name": "url",     "valueUri":    JDV_J02_URL},
        {"name": "system",  "valueUri":    TRE_R02_URL},
        {"name": "code",    "valueString": "SA03"},
        {"name": "display", "valueString": "Etablissement privé PSPH"},
    ],
}
vc_headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
r_vc_post = http_post(url_vc_post, body=vc_body, headers=vc_headers)
vc_post = r_vc_post.json()
save_artifact(70, "validate-sa03-result.json", vc_post)

# Étape 60 — Réception et intégration
params_p = {p["name"]: p for p in vc_post.get("parameter", [])}
result_p  = params_p.get("result",  {}).get("valueBoolean")
display_p = params_p.get("display", {}).get("valueString", "")

# Étape 70 — PREUVE
print("[PREUVE étape 70] $validate-code SA03 (POST) :")
print(f"  result  : {result_p}  (attendu: True)")
print(f"  display : {display_p}  (attendu: Etablissement privé PSPH)")
assert result_p is True, f"Validation échouée pour SA03"
print("  ✓ Code valide")

  → POST https://gazelle-proxypatans.kereval.cloud:10102/fhir/ValueSet/$validate-code
  ✓ runs/06_iti99_validate_20260311T212121/step070_validate-sa03-result.json
[PREUVE étape 70] $validate-code SA03 (POST) :
  result  : True  (attendu: True)
  display : Etablissement privé PSPH  (attendu: Etablissement privé PSPH)
  ✓ Code valide


---
## Étapes 80–110 — Validation GET : SA100 dans TRE-R02 (code invalide)

**Requête** : `GET /fhir/CodeSystem/$validate-code?url=...TRE-R02-SecteurActivite...&code=SA100&_format=json`
**Objectif** : Vérifier que `SA100` est absent du CodeSystem — réponse attendue : `result=false`.

In [24]:
# Étape 80 — Construction de la requête
# Étape 90 — TRANSACTION : GET $validate-code SA100 (code inexistant)
url_neg = f"{FHIR_BASE}/CodeSystem/$validate-code?url={quote(TRE_R02_URL)}&code=SA100&_format=json"
r_neg = http_get(url_neg)
neg = r_neg.json()
save_artifact(110, "validate-sa100-negative-result.json", neg)

# Étape 100 — Réception et intégration
params_n  = {p["name"]: p for p in neg.get("parameter", [])}
result_n  = params_n.get("result",  {}).get("valueBoolean")
message_n = params_n.get("message", {}).get("valueString", "")

# Étape 110 — PREUVE
print("[PREUVE étape 110] $validate-code SA100 (GET) — test négatif :")
print(f"  result  : {result_n}  (attendu: False)")
print(f"  message : {message_n}")
assert result_n is False, f"SA100 ne devrait pas être valide, résultat obtenu : {result_n}"
print("  ✓ Code correctement rejeté")


  → GET https://gazelle-proxypatans.kereval.cloud:10102/fhir/CodeSystem/$validate-code?url=https%3A//mos.esante.gouv.fr/NOS/TRE_R02-SecteurActivite/FHIR/TRE-R02-SecteurActivite&code=SA100&_format=json
  ✓ runs/06_iti99_validate_20260311T212121/step110_validate-sa100-negative-result.json
[PREUVE étape 110] $validate-code SA100 (GET) — test négatif :
  result  : False  (attendu: False)
  message : Unknown code 'SA100' in the CodeSystem 'https://mos.esante.gouv.fr/NOS/TRE_R02-SecteurActivite/FHIR/TRE-R02-SecteurActivite' version '20250523120000'
  ✓ Code correctement rejeté


---
## Récapitulatif

In [25]:
print(f"Run : {RUN_DIR}")
print(f"{'Fichier':<45} {'Taille':>10}")
print("-" * 57)
for fname in sorted(os.listdir(RUN_DIR)):
    fpath = os.path.join(RUN_DIR, fname)
    size  = os.path.getsize(fpath)
    size_str = f"{size/1024:.1f} KB" if size < 1_000_000 else f"{size/1024/1024:.1f} MB"
    print(f"  {fname:<43} {size_str:>10}")
print(f"\n✓ {NOTEBOOK_ID} — terminé.")


Run : runs/06_iti99_validate_20260311T212121
Fichier                                           Taille
---------------------------------------------------------
  step030_validate-sa01-result.json               0.5 KB
  step070_validate-sa03-result.json               0.5 KB
  step110_validate-sa100-negative-result.json     1.3 KB

✓ 06_iti99_validate — terminé.
